# Separate free Gemini dev comparison
No Groq, no holdout. One trial per task/version, exploratory only. Stop at request budget.

In [ ]:
# Separate Gemini exploratory comparison. Never executes Groq or holdout.
from pathlib import Path
import os,sys,subprocess,json,time,zipfile,base64,hashlib
from google.colab import userdata
ROOT=Path('/content/agent-evals-gemini')
if not ROOT.exists():
    subprocess.run(['git','clone','https://github.com/yuvasree023/agent-evals.git',str(ROOT)],check=True)
else:
    subprocess.run(['git','-C',str(ROOT),'fetch','origin'],check=True)
    subprocess.run(['git','-C',str(ROOT),'reset','--hard','origin/main'],check=True)
os.chdir(ROOT)
print('GEMINI_SOURCE',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())
subprocess.run(['python3','-m','unittest','discover','-s','tests'],check=True)
subprocess.run(['python3','run.py','--validate'],check=True)
os.environ['GEMINI_API_KEY']=userdata.get('GEMINI_API_KEY')
sys.path.insert(0,str(ROOT))
import run,providers,agent,agent_v2,agent_candidates
import graders_v2 as g2
assert 'raw' in Path('agent_candidates.py').read_text()
model='gemini-3.5-flash-lite'
# One fresh trial/task/version, interleaved to limit quota-related imbalance.
# This is exploratory, not the four-trial M1 or an improvement claim.
tasks=[t for t in run.load_tasks() if t['split']=='dev']
versions=['v1','v2','v3','v4']; records=[]; budget={'calls':0}; CAP=400
original_chat=providers.chat
class ComparisonBudgetReached(Exception): pass
def counted_chat(*args,**kwargs):
    if budget['calls']>=CAP: raise ComparisonBudgetReached('Stopped at 400 request budget; no paid fallback.')
    budget['calls']+=1
    return original_chat(*args,**kwargs)
providers.chat=counted_chat
out=ROOT/'results/local';out.mkdir(parents=True,exist_ok=True)
path=out/'gemini-dev-candidates-2026-10-13.jsonl'
stop=None
try:
    for task in tasks:
        for version in versions:
            start=time.monotonic()
            fn=agent.run_task if version=='v1' else agent_v2.run_task if version=='v2' else agent_candidates.run_task
            kw={'version':version} if version in {'v3','v4'} else {}
            answer,env,messages=fn(task,'gemini',model,use_cache=False,**kw)
            before=run.task_state(task)
            rec={'task_id':task['id'],'trial':1,'agent_version':version,'provider':'gemini','model':model,'input':task['input'],'answer':answer,'trace':env.trace,'before':before,'after':env.state(),'messages':run._strip(messages),'latency_ms':round((time.monotonic()-start)*1000),'cost_usd':0}
            checks,viol=g2.grade_v2(task,before,rec)
            rec.update(grader_v2=checks,violations=viol,attempt_findings=agent_candidates.trajectory_findings(task['input'],env.trace),response_findings=agent_candidates.response_findings(task['input'],answer,env.trace))
            records.append(rec)
            with path.open('a') as f:f.write(json.dumps(rec)+'\n')
            print('GEMINI_PROGRESS',len(records),task['id'],version,flush=True)
            env.db.close()
except Exception as exc:
    stop=type(exc).__name__+': '+str(exc)[:600]
    print('GEMINI_STOP',stop,flush=True)
finally:
    summary={'model':model,'provider':'gemini','protocol':'35 dev tasks x four versions x one uncached trial, interleaved; exploratory only','expected_records':140,'records':len(records),'requests_started':budget['calls'],'request_cap':CAP,'complete':len(records)==140,'stop':stop,'cost_usd':0,'holdout_executed':False}
    (out/'gemini-dev-candidates-2026-10-13-summary.json').write_text(json.dumps(summary,indent=2))
    archive=Path('/content/gemini-dev-candidates-2026-10-13.zip')
    with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
        for p in out.glob('gemini-dev-candidates-2026-10-13*'):z.write(p,p.name)
    print('GEMINI_FINAL',json.dumps(summary),flush=True)
    print('GEMINI_ARCHIVE_B64_START')
    print(base64.b64encode(archive.read_bytes()).decode())
    print('GEMINI_ARCHIVE_B64_END')
